# 🏥 OmniMedical Suite - نظام OCR هجين مع تصحيح إملائي

دفتر ملاحظات تفاعلي لنظام التعرف الضوئي على الحروف (OCR) المتعدد المحركات مع التصحيح الإملائي الذكي للنصوص الطبية العربية.

### المزايا الرئيسية:
- **4 محركات OCR**: Tesseract, EasyOCR, PaddleOCR, TrOCR
- **5 استراتيجيات دمج**: تصويت مرجّح، إجماع حرفي، أفضل ثقة، تراجع ذكي، كل المحركات
- **تصحيح إملائي** عبر نموذج AraBART GEC (CAMeL-Lab)
- **إزالة التكرار الدلالي** باستخدام Sentence-BERT + FAISS
- **استخراج الكيانات الطبية** بالتعبيرات النمطية
- **واجهة Gradio تفاعلية** مع تصدير بيانات التدريب (RLHF)

---

> ⚡ **يُوصى بتشغيله على Google Colab مع مسرّع GPU (T4 أو أفضل)**


In [ ]:
# Install system packages
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-ara tesseract-ocr-eng poppler-utils > /dev/null 2>&1

# Install Python packages
!pip install -q pytesseract easyocr paddleocr sentence-transformers faiss-cpu hdbscan
!pip install -q opencv-python-headless matplotlib transformers torch accelerate arabic-reshaper python-bidi gradio


## 2. تحميل محركات OCR والنماذج

تحميل جميع المحركات والنماذج المستخدمة في النظام مع التعامل مع حالات الفشل.


In [ ]:
import os
import re
import json
import warnings
import numpy as np
import cv2
import pytesseract
import easyocr
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter
from typing import List, Dict, Tuple, Optional

warnings.filterwarnings('ignore')

# ── Track loaded engines ──────────────────────────────────────
loaded_engines = {}

# 1) Tesseract
try:
    pytesseract.get_tesseract_version()
    loaded_engines['tesseract'] = True
    print('✅ Tesseract loaded')
except Exception as e:
    loaded_engines['tesseract'] = False
    print(f'❌ Tesseract failed: {e}')

# 2) EasyOCR
try:
    easyocr_reader = easyocr.Reader(['ar', 'en'], gpu=True, verbose=False)
    loaded_engines['easyocr'] = True
    print('✅ EasyOCR loaded')
except Exception as e:
    easyocr_reader = None
    loaded_engines['easyocr'] = False
    print(f'❌ EasyOCR failed: {e}')

# 3) PaddleOCR
try:
    from paddleocr import PaddleOCR
    paddle_reader = PaddleOCR(use_angle_cls=True, lang='ar', show_log=False)
    loaded_engines['paddleocr'] = True
    print('✅ PaddleOCR loaded')
except Exception as e:
    paddle_reader = None
    loaded_engines['paddleocr'] = False
    print(f'❌ PaddleOCR failed: {e}')

# 4) TrOCR (HuggingFace)
trocr_processor = trocr_model = None
try:
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    trocr_processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
    trocr_model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')
    trocr_model.eval()
    loaded_engines['trocr'] = True
    print('✅ TrOCR loaded')
except Exception as e:
    loaded_engines['trocr'] = False
    print(f'❌ TrOCR failed: {e}')

# 5) AraBART GEC model
gec_tokenizer = gec_model = None
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    gec_tokenizer = AutoTokenizer.from_pretrained('CAMeL-Lab/arabart-qalb14-gec-ged-13')
    gec_model = AutoModelForSeq2SeqLM.from_pretrained('CAMeL-Lab/arabart-qalb14-gec-ged-13')
    gec_model.eval()
    loaded_engines['gec'] = True
    print('✅ AraBART GEC loaded')
except Exception as e:
    loaded_engines['gec'] = False
    print(f'❌ AraBART GEC failed: {e}')

# 6) SentenceTransformer for semantic dedup
embedder = None
try:
    from sentence_transformers import SentenceTransformer
    import faiss
    embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    loaded_engines['embedder'] = True
    print('✅ SentenceTransformer + FAISS loaded')
except Exception as e:
    loaded_engines['embedder'] = False
    print(f'❌ SentenceTransformer failed: {e}')

print(f'\n📊 Summary: {sum(loaded_engines.values())}/{len(loaded_engines)} engines loaded')
print('Available:', [k for k, v in loaded_engines.items() if v])


## 4. دوال المعالجة

جميع الدوال المستخدمة في المعالجة المسبقة، التعرّف، الدمج، التصحيح، والإزالة الدلالية للتكرار.


In [ ]:
# ══════════════════════════════════════════════════════════════
#  IMAGE PREPROCESSING
# ══════════════════════════════════════════════════════════════
def preprocess_image(image: np.ndarray) -> np.ndarray:
    """Full preprocessing pipeline: grayscale → denoise → CLAHE → Otsu."""
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()
    # Denoise
    denoised = cv2.fastNlMeansDenoising(gray, None, h=10)
    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    # Otsu binarization
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


# ══════════════════════════════════════════════════════════════
#  INDIVIDUAL OCR ENGINES
# ══════════════════════════════════════════════════════════════
def run_tesseract(image: np.ndarray) -> Dict:
    """Run Tesseract with Arabic + English."""
    if not loaded_engines.get('tesseract'):
        return {'text': '', 'confidence': 0.0, 'engine': 'tesseract'}
    try:
        processed = preprocess_image(image)
        data = pytesseract.image_to_data(processed, lang='ara+eng', output_type=pytesseract.Output.DICT)
        texts, confs = [], []
        for t, c in zip(data['text'], data['conf']):
            t = t.strip()
            if t and int(c) > 0:
                texts.append(t)
                confs.append(float(c))
        combined = ' '.join(texts)
        avg_conf = float(np.mean(confs)) if confs else 0.0
        return {'text': combined, 'confidence': avg_conf / 100.0, 'engine': 'tesseract'}
    except Exception as e:
        return {'text': '', 'confidence': 0.0, 'engine': 'tesseract', 'error': str(e)}


def run_easyocr(image: np.ndarray) -> Dict:
    """Run EasyOCR with Arabic + English."""
    if not loaded_engines.get('easyocr'):
        return {'text': '', 'confidence': 0.0, 'engine': 'easyocr'}
    try:
        results = easyocr_reader.readtext(image)
        texts, confs = [], []
        for (bbox, txt, conf) in results:
            if txt.strip():
                texts.append(txt.strip())
                confs.append(float(conf))
        combined = ' '.join(texts)
        avg_conf = float(np.mean(confs)) if confs else 0.0
        return {'text': combined, 'confidence': avg_conf, 'engine': 'easyocr'}
    except Exception as e:
        return {'text': '', 'confidence': 0.0, 'engine': 'easyocr', 'error': str(e)}


def run_paddleocr(image: np.ndarray) -> Dict:
    """Run PaddleOCR."""
    if not loaded_engines.get('paddleocr'):
        return {'text': '', 'confidence': 0.0, 'engine': 'paddleocr'}
    try:
        result = paddle_reader.ocr(image, cls=True)
        texts, confs = [], []
        if result and result[0]:
            for line in result[0]:
                if len(line) >= 2:
                    texts.append(line[1][0])
                    confs.append(float(line[1][1]))
        combined = ' '.join(texts)
        avg_conf = float(np.mean(confs)) if confs else 0.0
        return {'text': combined, 'confidence': avg_conf, 'engine': 'paddleocr'}
    except Exception as e:
        return {'text': '', 'confidence': 0.0, 'engine': 'paddleocr', 'error': str(e)}


def run_trocr(image: np.ndarray) -> Dict:
    """Run TrOCR (handles one line at a time; uses full image)."""
    if not loaded_engines.get('trocr'):
        return {'text': '', 'confidence': 0.0, 'engine': 'trocr'}
    try:
        import torch
        pil_img = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        pixel_values = trocr_processor(pil_img, return_tensors='pt').pixel_values
        with torch.no_grad():
            generated_ids = trocr_model.generate(pixel_values)
        text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return {'text': text, 'confidence': 1.0, 'engine': 'trocr'}
    except Exception as e:
        return {'text': '', 'confidence': 0.0, 'engine': 'trocr', 'error': str(e)}


# ══════════════════════════════════════════════════════════════
#  ENSEMBLE FUSION STRATEGIES
# ══════════════════════════════════════════════════════════════
def weighted_voting_fusion(results: List[Dict]) -> Dict:
    """Fuse results using weighted confidence voting."""
    weights = {'tesseract': 0.7, 'easyocr': 0.9, 'paddleocr': 0.85, 'trocr': 0.95}
    weighted_texts = []
    for r in results:
        if r['text'].strip():
            w = weights.get(r['engine'], 0.5) * r['confidence']
            weighted_texts.append((r['text'], w))
    if not weighted_texts:
        return {'text': '', 'confidence': 0.0, 'strategy': 'weighted_voting'}
    # Sort by weight descending
    weighted_texts.sort(key=lambda x: x[1], reverse=True)
    best_text = weighted_texts[0][0]
    avg_conf = float(np.mean([w for _, w in weighted_texts]))
    return {'text': best_text, 'confidence': avg_conf, 'strategy': 'weighted_voting'}


def character_level_consensus(results: List[Dict]) -> Dict:
    """Fuse by majority-voting at character level."""
    valid = [r['text'] for r in results if r['text'].strip()]
    if not valid:
        return {'text': '', 'confidence': 0.0, 'strategy': 'char_consensus'}
    # Pad to same length
    max_len = max(len(t) for t in valid)
    padded = [t.ljust(max_len) for t in valid]
    consensus = []
    for i in range(max_len):
        chars = [p[i] for p in padded]
        most_common = Counter(chars).most_common(1)[0]
        consensus.append(most_common[0])
    text = ''.join(consensus).rstrip()
    return {'text': text, 'confidence': 0.8, 'strategy': 'char_consensus'}


def best_confidence(results: List[Dict]) -> Dict:
    """Return result with highest confidence."""
    valid = [r for r in results if r['text'].strip()]
    if not valid:
        return {'text': '', 'confidence': 0.0, 'strategy': 'best_confidence'}
    best = max(valid, key=lambda r: r['confidence'])
    best['strategy'] = 'best_confidence'
    return best


def smart_fallback(results: List[Dict]) -> Dict:
    """Try TrOCR first (best for handwritten), fall back to EasyOCR, then PaddleOCR, then Tesseract."""
    priority = ['trocr', 'easyocr', 'paddleocr', 'tesseract']
    by_engine = {r['engine']: r for r in results if r['text'].strip()}
    for eng in priority:
        if eng in by_engine:
            result = by_engine[eng].copy()
            result['strategy'] = 'smart_fallback'
            return result
    return {'text': '', 'confidence': 0.0, 'strategy': 'smart_fallback'}


def ensemble_ocr(image: np.ndarray, strategy: str = 'weighted_voting') -> Dict:
    """Run all available engines and fuse results."""
    # Run all engines
    results = []
    if loaded_engines.get('tesseract'):
        results.append(run_tesseract(image))
    if loaded_engines.get('easyocr'):
        results.append(run_easyocr(image))
    if loaded_engines.get('paddleocr'):
        results.append(run_paddleocr(image))
    if loaded_engines.get('trocr'):
        results.append(run_trocr(image))

    # Fuse
    strategies = {
        'weighted_voting': weighted_voting_fusion,
        'char_consensus': character_level_consensus,
        'best_confidence': best_confidence,
        'smart_fallback': smart_fallback,
    }
    if strategy == 'all_engines':
        return {
            'results_per_engine': results,
            'strategy': 'all_engines'
        }
    fusion_fn = strategies.get(strategy, weighted_voting_fusion)
    return fusion_fn(results)


# ══════════════════════════════════════════════════════════════
#  GRAMMAR ERROR CORRECTION (GEC)
# ══════════════════════════════════════════════════════════════
def correct_with_gec(text: str) -> str:
    """Correct Arabic text using AraBART GEC model."""
    if not loaded_engines.get('gec') or not text.strip():
        return text
    try:
        import torch
        inputs = gec_tokenizer(text, return_tensors='pt', max_length=128, truncation=True)
        with torch.no_grad():
            generated = gec_model.generate(**inputs, max_length=128, num_beams=5)
        corrected = gec_tokenizer.decode(generated[0], skip_special_tokens=True)
        return corrected
    except Exception as e:
        print(f'GEC error: {e}')
        return text


# ══════════════════════════════════════════════════════════════
#  SEMANTIC DEDUPLICATION
# ══════════════════════════════════════════════════════════════
def semantic_dedup(chunks: List[str], threshold: float = 0.85) -> List[str]:
    """Remove semantically duplicate chunks using FAISS."""
    if not loaded_engines.get('embedder') or len(chunks) <= 1:
        return chunks
    try:
        embeddings = embedder.encode(chunks, convert_to_numpy=True)
        dim = embeddings.shape[1]
        index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(embeddings)
        index.add(embeddings)
        # Search
        D, I = index.search(embeddings, min(10, len(chunks)))
        to_remove = set()
        for i in range(len(chunks)):
            for j_idx, (sim, neighbor) in enumerate(zip(D[i], I[i])):
                if j_idx == 0:
                    continue  # self
                if sim >= threshold and i < neighbor:
                    to_remove.add(neighbor)
        unique = [chunks[i] for i in range(len(chunks)) if i not in to_remove]
        return unique
    except Exception as e:
        print(f'Semantic dedup error: {e}')
        return chunks


# ══════════════════════════════════════════════════════════════
#  MEDICAL ENTITY EXTRACTION (Regex-based NER)
# ══════════════════════════════════════════════════════════════
MEDICAL_PATTERNS = {
    'medication': r'(?:قرص|كبسولة|ملعقة|حقنة|قطرة|مرهم|شراب)\s+\S+',
    'dosage': r'\d+\s*(?:مغ|ملجم|جم|مل|وحدة)',
    'frequency': r'(?:مرة|مرتين|ثلاث|أربع)\s+(?:يوميا|أسبوعيا|شهريا|في اليوم)',
    'diagnosis': r'(?:مرض|تشخيص|حالة)\s+[:\-]?\s*\S+(?:\s+\S+){0,3}',
    'patient_info': r'(?:اسم المريض|رقم الملف|تاريخ الولادة|الجنس|الفصيلة)\s+[:\-]?\s*\S+(?:\s+\S+){0,2}',
    'date': r'\d{1,4}[/-]\d{1,2}[/-]\d{1,4}',
    'doctor': r'(?:د\.?|طبيب|أستاذ|دكتور)\s+\S+(?:\s+\S+){0,3}',
    'lab_test': r'(?:تحليل|فحص|صورة|أشعة)\s+\S+(?:\s+\S+){0,3}',
}


def extract_medical_entities(text: str) -> Dict[str, List[str]]:
    """Extract Arabic medical entities using regex patterns."""
    entities = {}
    for entity_type, pattern in MEDICAL_PATTERNS.items():
        matches = re.findall(pattern, text)
        if matches:
            entities[entity_type] = list(set(matches))
    return entities


print('✅ All processing functions defined successfully')


## 6. المعالجة التفاعلية

واجهة Gradio التفاعلية مع 4 تبويبات: رفع ومعالجة، المقارنة والتصحيح، تصدير بيانات التدريب، واختبار سريع.


In [ ]:
import gradio as gr
from datetime import datetime

# ── Feedback storage ──────────────────────────────────────────
feedback_log = []

# ── Arabic medical examples ──────────────────────────────────
EXAMPLES = [
    {
        'name': 'وصفة دوائية',
        'text': 'المريض: أحمد محمد العلي\nالوصفة: قرص باراسيتامول 500 مغ مرتين يوميا بعد الأكل\nالمدة: 7 أيام\nالتاريخ: 2024/03/15\nالتوقيع: د. سارة الأحمد'
    },
    {
        'name': 'تقرير مختبر',
        'text': 'اسم المريض: فاطمة خالد\nرقم الملف: 45231\nتحليل الدم: الهيموغلوبين 12.5 جم/ديسيلتر\nالكرات البيضاء 7000 خلية/ميكرولتر\nالصفيحات 250000\nالتاريخ: 2024/05/20'
    },
    {
        'name': 'تقرير أشعة',
        'text': 'صورة شعاعية للصدر\nالحالة: التهاب رئوي في الفص السفلي الأيمن\nالتوصية: مراجعة بعد أسبوعين\nالطبيب: د. محمد الراشد'
    },
    {
        'name': 'تقرير طبي عام',
        'text': 'التشخيص: ارتفاع ضغط الدم المزمن\nالعلاج: قرص أملوديبين 5 مغ مرة واحدة يوميا\nمتابعة: فحص وظائف الكلى بعد شهر\nالطبيب: أستاذ عبدالله المنصور'
    },
    {
        'name': 'وصفة أطفال',
        'text': 'الطفل: نورة سعد\nالعمر: 3 سنوات\nالوزن: 14 كجم\nالدواء: شراب أموكسيسيلين 125 مغ / 5 مل\nالجرعة: 5 مل ثلاث مرات يوميا\nالمدة: 10 أيام'
    },
    {
        'name': 'تقرير عيون',
        'text': 'فحص النظر\nالعيون اليمنى: -2.50\nالعيون اليسرى: -2.75\nالقرنية: طبيعية\nالتوصية: نظارات طبية جديدة'
    },
]


# ── Tab 1: Upload & Process ───────────────────────────────────
def process_upload(image, strategy, use_trocr, use_gec, use_dedup):
    if image is None:
        return '⚠️ الرجاء رفع صورة أولاً', '', ''
    img = np.array(image)
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    # Run ensemble
    result = ensemble_ocr(img, strategy)
    if 'results_per_engine' in result:
        raw_text = '\n'.join(
            f"【{r['engine']}】 ({r['confidence']:.0%}): {r['text']}"
            for r in result['results_per_engine'] if r['text'].strip()
        )
        final_text = max(result['results_per_engine'], key=lambda r: r['confidence'])['text']
    else:
        raw_text = result['text']
        final_text = result['text']

    # Optional TrOCR
    if use_trocr and loaded_engines.get('trocr'):
        trocr_result = run_trocr(img)
        if trocr_result['text'].strip():
            raw_text += f"\n【trocr】 {trocr_result['text']}"

    # Optional GEC
    corrected = final_text
    if use_gec:
        corrected = correct_with_gec(final_text)

    # Optional Dedup
    dedup_info = ''
    if use_dedup:
        chunks = [line.strip() for line in corrected.split('\n') if line.strip()]
        unique = semantic_dedup(chunks)
        dedup_info = f'\n\n🔍 إزالة التكرار: {len(chunks)} → {len(unique)} سطر'
        corrected = '\n'.join(unique)

    # Extract entities
    entities = extract_medical_entities(corrected)
    entities_str = '\n'.join(f'  • {k}: {v}' for k, v in entities.items()) if entities else 'لا توجد كيانات مميزة'

    summary = f'📊 النص المستخرج:\n{final_text}\n\n✏️ بعد التصحيح:\n{corrected}{dedup_info}\n\n🏥 الكيانات الطبية:\n{entities_str}'

    # Store for feedback
    feedback_log.append({
        'timestamp': datetime.now().isoformat(),
        'strategy': strategy,
        'raw_text': final_text,
        'corrected_text': corrected,
        'entities': entities
    })

    return summary, corrected, raw_text


# ── Tab 2: Compare & Correct ──────────────────────────────────
def compare_and_rate(original, corrected, rating, feedback_text):
    entry = {
        'timestamp': datetime.now().isoformat(),
        'original': original,
        'corrected': corrected,
        'rating': int(rating) if rating else 0,
        'feedback': feedback_text
    }
    feedback_log.append(entry)
    return f'✅ تم حفظ التقييم (التقييم: {"⭐" * (int(rating) if rating else 0)})\nإجمالي التقييمات المحفوظة: {len(feedback_log)}'


# ── Tab 4: Quick Test ─────────────────────────────────────────
def quick_test(example_idx):
    ex = EXAMPLES[int(example_idx)]
    # Simulate OCR correction
    corrected = correct_with_gec(ex['text'])
    entities = extract_medical_entities(corrected)
    entities_str = '\n'.join(f'  • {k}: {v}' for k, v in entities.items()) if entities else 'لا توجد كيانات مميزة'
    return f'📋 المثال: {ex["name"]}\n\n📝 النص الأصلي:\n{ex["text"]}\n\n✏️ بعد التصحيح:\n{corrected}\n\n🏥 الكيانات الطبية:\n{entities_str}'


# ══════════════════════════════════════════════════════════════
#  BUILD GRADIO INTERFACE
# ══════════════════════════════════════════════════════════════
with gr.Blocks(title='OmniMedical OCR', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🏥 OmniMedical Suite - نظام OCR هجين')

    with gr.Tab('📤 رفع ومعالجة'):
        with gr.Row():
            with gr.Column(scale=1):
                img_input = gr.Image(type='pil', label='رفع صورة')
                strategy_dd = gr.Dropdown(
                    choices=['weighted_voting', 'char_consensus', 'best_confidence', 'smart_fallback', 'all_engines'],
                    value='weighted_voting', label='استراتيجية الدمج'
                )
                with gr.Row():
                    chk_trocr = gr.Checkbox(label='TrOCR (يدوي)', value=False)
                    chk_gec = gr.Checkbox(label='تصحيح إملائي GEC', value=True)
                    chk_dedup = gr.Checkbox(label='إزالة التكرار الدلالي', value=True)
                btn_process = gr.Button('🚀 معالجة', variant='primary')
            with gr.Column(scale=2):
                txt_summary = gr.Textbox(label='الملخص', lines=10)
                txt_corrected = gr.Textbox(label='النص المصحّح', lines=6)
                txt_raw = gr.Textbox(label='النتائج الخام لكل محرك', lines=6)

        btn_process.click(
            process_upload,
            inputs=[img_input, strategy_dd, chk_trocr, chk_gec, chk_dedup],
            outputs=[txt_summary, txt_corrected, txt_raw]
        )

    with gr.Tab('⚖️ المقارنة والتصحيح'):
        with gr.Row():
            txt_original_cmp = gr.Textbox(label='النص الأصلي (قبل التصحيح)', lines=6)
            txt_corrected_cmp = gr.Textbox(label='النص المصحّح (بعد التصحيح)', lines=6)
        rating_slider = gr.Slider(minimum=1, maximum=5, step=1, value=3, label='التقييم (1-5 نجوم)')
        txt_feedback = gr.Textbox(label='ملاحظات إضافية', lines=3)
        btn_rate = gr.Button('💾 حفظ التقييم')
        txt_rate_result = gr.Textbox(label='النتيجة')

        btn_rate.click(
            compare_and_rate,
            inputs=[txt_original_cmp, txt_corrected_cmp, rating_slider, txt_feedback],
            outputs=[txt_rate_result]
        )

    with gr.Tab('💾 تصدير بيانات التدريب'):
        gr.Markdown('### تصدير بيانات التقييم والتصحيح للاستخدام في تدريب RLHF')
        btn_json = gr.Button('📥 تصدير feedback.json')
        btn_rlhf = gr.Button('📥 تصدير rlhf_training_data.jsonl')
        btn_csv = gr.Button('📥 تصدير training_feedback.csv')
        file_json = gr.File(label='feedback.json')
        file_rlhf = gr.File(label='rlhf_training_data.jsonl')
        file_csv = gr.File(label='training_feedback.csv')

    with gr.Tab('⚡ اختبار سريع'):
        gr.Markdown('### اختبار النظام على أمثلة طبية عربية')
        example_names = [ex['name'] for ex in EXAMPLES]
        example_dd = gr.Dropdown(choices=example_names, label='اختر مثالاً')
        btn_test = gr.Button('▶️ تشغيل الاختبار', variant='primary')
        txt_test_result = gr.Textbox(label='النتيجة', lines=15)

        btn_test.click(quick_test, inputs=[example_dd], outputs=[txt_test_result])

# Launch
demo.launch(share=True, debug=False)


## 8. تصدير بيانات التدريب

دوال تصدير بيانات التقييم بصيغ JSON و JSONL و CSV للاستخدام في تدريب نماذج RLHF.


In [ ]:
import csv
import json
from pathlib import Path
from datetime import datetime

feedback_log = []  # global feedback storage


def export_to_json(filepath: str = 'feedback.json') -> str:
    """Export all feedback as a JSON file."""
    data = {
        'export_date': datetime.now().isoformat(),
        'total_entries': len(feedback_log),
        'entries': feedback_log
    }
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return filepath


def export_to_rlhf(filepath: str = 'rlhf_training_data.jsonl') -> str:
    """Export feedback in RLHF format (JSONL with prompt/chosen/rejected)."""
    with open(filepath, 'w', encoding='utf-8') as f:
        for entry in feedback_log:
            original = entry.get('raw_text') or entry.get('original', '')
            corrected = entry.get('corrected_text') or entry.get('corrected', '')
            if original.strip() and corrected.strip():
                rlhf_entry = {
                    'prompt': f'صحّح النص الطبي التالي:\n{original}',
                    'chosen': corrected,
                    'rejected': original,
                    'rating': entry.get('rating', 0),
                    'source': 'omni-medical-ocr'
                }
                f.write(json.dumps(rlhf_entry, ensure_ascii=False) + '\n')
    return filepath


def export_to_csv(filepath: str = 'training_feedback.csv') -> str:
    """Export feedback as CSV."""
    if not feedback_log:
        return ''
    fieldnames = ['timestamp', 'strategy', 'original', 'corrected', 'rating', 'feedback']
    with open(filepath, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        for entry in feedback_log:
            row = {
                'timestamp': entry.get('timestamp', ''),
                'strategy': entry.get('strategy', ''),
                'original': entry.get('raw_text') or entry.get('original', ''),
                'corrected': entry.get('corrected_text') or entry.get('corrected', ''),
                'rating': entry.get('rating', ''),
                'feedback': entry.get('feedback', ''),
            }
            writer.writerow(row)
    return filepath


# ── Generate exports ──────────────────────────────────────────
json_path = export_to_json('/content/feedback.json')
rlhf_path = export_to_rlhf('/content/rlhf_training_data.jsonl')
csv_path = export_to_csv('/content/training_feedback.csv')

print(f'📁 feedback.json: {json_path}')
print(f'📁 rlhf_training_data.jsonl: {rlhf_path}')
print(f'📁 training_feedback.csv: {csv_path}')
print(f'\n📊 Total feedback entries: {len(feedback_log)}')

# Download buttons for Colab
from google.colab import files
if feedback_log:
    print('\n⬇️ تحميل الملفات:')
    files.download(json_path)
